# Track A — Data & Model (FIR Categorization)
**Faculty ask: train on 2 datasets (ILSI + a second one), run each model 10x, report mean ± std.**

Order:
1. Dataset 1 (ILSI): download, EDA, label freeze
2. Baseline (TF-IDF+RF), BiLSTM, DistilBERT — each wrapped as a function, run N times, averaged
3. Export the single best `predict()` + threshold -> hand to Track B
4. Dataset 2 (ACI — a second, independent Indian criminal-law dataset): same 3 models, same averaging
5. Combined summary table for both datasets, for the deck / faculty review
6. Explainability (optional, skip if short on time)

**Honest note on Dataset 2:** ILSI (what you already used) IS the standard large public
IPC-section-labelled dataset from this research group (IIT Kharagpur, Law-AI lab) — there
isn't a second dataset of that size and exact same 100-section label format publicly available.
The best real, freely downloadable second option is **ACI** (COLING 2020, same lab): 189 total
documents, IPC *charge names* (e.g. "murder", "cheating") rather than numeric section codes,
and no login/gating required. It's much smaller, so treat its numbers as a generalisation check
on a different label taxonomy — not as a like-for-like replacement for ILSI's scale. This is a
completely normal and defensible thing to present: 'we validated the same architecture on a second,
independently-annotated Indian legal dataset to show it isn't just overfit to one label set.'

In [ ]:
# Lab machine: packages are installed once with setup_lab.sh (see README_LAB.md).
# Nothing to install here. If an import fails later, run in a terminal:  pip install -r requirements.txt

In [ ]:
import json, os, re, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, jaccard_score, precision_score, recall_score

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: no GPU visible to PyTorch. Check nvidia-smi and that the CUDA build of torch is installed, then restart the kernel.")
else:
    print("GPU:", torch.cuda.get_device_name(0))

# Use the local DistilBERT folder if it was copied in (lab internet may be blocked); otherwise download from the Hub.
LOCAL_DISTILBERT = "./models/distilbert-base-uncased"
DISTILBERT_NAME = LOCAL_DISTILBERT if os.path.isdir(LOCAL_DISTILBERT) else "distilbert-base-uncased"
print("DistilBERT weights from:", DISTILBERT_NAME)

def tokenize(text):
    return re.findall(r"\w+", text.lower())

def average_runs(metric_dicts):
    """[{'macro_f1':..,'micro_f1':..}, ...] over N runs -> {'macro_f1': {'mean':.., 'std':..}, ...}"""
    keys = metric_dicts[0].keys()
    out = {}
    for k in keys:
        vals = [d[k] for d in metric_dicts]
        out[k] = {"mean": float(np.mean(vals)), "std": float(np.std(vals))}
    return out

def print_avg(name, avg):
    parts = [f"{k}={v['mean']:.4f}±{v['std']:.4f}" for k, v in avg.items()]
    print(f"{name}: " + "  ".join(parts))

# ---------- crash/logout-safe runs ----------
# Every finished run is written to ./checkpoints/<tag>.json immediately. If the process is killed
# (logout, reboot, lab closing), starting it again skips the finished runs and continues.
CKPT_DIR = "./checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

def _ckpt_path(tag):
    return os.path.join(CKPT_DIR, tag + ".json")

def _ckpt_load(tag):
    path = _ckpt_path(tag)
    if not os.path.exists(path):
        return {}
    try:
        with open(path) as f:
            return json.load(f)
    except Exception as e:
        print(f"  WARNING: checkpoint {path} unreadable ({e}); starting this model's runs from scratch", flush=True)
        return {}

def ckpt_forget(tag, seed):
    done = _ckpt_load(tag)
    if str(seed) in done:
        del done[str(seed)]
        tmp = _ckpt_path(tag) + ".tmp"
        with open(tmp, "w") as f:
            json.dump(done, f)
        os.replace(tmp, _ckpt_path(tag))

def run_with_ckpt(tag, n_runs, fn, label):
    """Run fn(seed) -> metrics dict for seeds 0..n_runs-1, skipping seeds already in the checkpoint."""
    done = _ckpt_load(tag)
    for s in range(n_runs):
        if str(s) in done:
            print(f"  {label} seed {s}: already done (macro_f1={done[str(s)]['macro_f1']:.4f}), skipped", flush=True)
            continue
        t0 = time.time()
        m = fn(s)
        done[str(s)] = {k: float(v) for k, v in m.items()}
        tmp = _ckpt_path(tag) + ".tmp"
        with open(tmp, "w") as f:
            json.dump(done, f)
        os.replace(tmp, _ckpt_path(tag))   # atomic: a kill mid-write cannot corrupt the checkpoint
        print(f"  {label} seed {s}: macro_f1={m['macro_f1']:.4f}  ({time.time()-t0:.0f}s)", flush=True)
    return [done[str(s)] for s in range(n_runs)]

## 1. Get the dataset

Zenodo record: https://doi.org/10.5281/zenodo.6053791

**Lab version:** the three ILSI files are read from `./data/ILSI/`. If they are already there (copied from the pendrive / `download_data.py`), nothing is downloaded. If not, this cell tries to download them from Zenodo.


In [ ]:
import urllib.request

DATA_DIR = "./data/ILSI"
os.makedirs(DATA_DIR, exist_ok=True)

FILES = {
    "train.jsonl": "https://zenodo.org/records/6053791/files/train.jsonl?download=1",
    "dev.jsonl":   "https://zenodo.org/records/6053791/files/dev.jsonl?download=1",
    "test.jsonl":  "https://zenodo.org/records/6053791/files/test.jsonl?download=1",
}

for fname, url in FILES.items():
    fpath = os.path.join(DATA_DIR, fname)
    if os.path.exists(fpath) and os.path.getsize(fpath) > 0:
        print(fname, "- already present, skipping")
    else:
        print("downloading", fname, "...")
        try:
            urllib.request.urlretrieve(url, fpath)
        except Exception as e:
            raise RuntimeError(f"Could not download {fname} ({e}). Copy train.jsonl, dev.jsonl, test.jsonl into {DATA_DIR} and re-run this cell.")
        print("done:", fname, "-", os.path.getsize(fpath) // (1024 * 1024), "MB")

print(sorted(os.listdir(DATA_DIR)))

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_raw = load_jsonl(os.path.join(DATA_DIR, "train.jsonl"))
dev_raw   = load_jsonl(os.path.join(DATA_DIR, "dev.jsonl"))
test_raw  = load_jsonl(os.path.join(DATA_DIR, "test.jsonl"))

print(len(train_raw), len(dev_raw), len(test_raw))
print(train_raw[0].keys())

### 2. EDA -> label freeze

Post the final label count to the group the moment this finishes.

In [ ]:
def to_df(raw):
    return pd.DataFrame({
        "id": [r["id"] for r in raw],
        "text": [" ".join(r["text"]) if isinstance(r["text"], list) else r["text"] for r in raw],
        "labels": [r["labels"] for r in raw],
    })

train_df = to_df(train_raw)
dev_df   = to_df(dev_raw)
test_df  = to_df(test_raw)

for name, df in [("train", train_df), ("dev", dev_df), ("test", test_df)]:
    print(name, "docs:", len(df))

all_labels = Counter(l for labs in train_df["labels"] for l in labs)
print("total unique sections:", len(all_labels))

In [ ]:
def filter_labels(df, min_freq, allowed=None):
    if allowed is None:
        counts = Counter(l for labs in df["labels"] for l in labs)
        allowed = {l for l, c in counts.items() if c >= min_freq}
    kept = df["labels"].apply(lambda labs: [l for l in labs if l in allowed])
    retained = (kept.apply(len) > 0).mean()
    return allowed, kept, retained

min_freq = 200
allowed_labels, train_kept, retention = filter_labels(train_df, min_freq)
print(f"min_freq={min_freq} -> {len(allowed_labels)} labels, retention={retention:.3f}")

if retention < 0.85:
    min_freq = 100
    allowed_labels, train_kept, retention = filter_labels(train_df, min_freq)
    print(f"adjusted min_freq={min_freq} -> {len(allowed_labels)} labels, retention={retention:.3f}")

train_df["labels"] = train_kept
train_df = train_df[train_df["labels"].apply(len) > 0].reset_index(drop=True)

_, dev_df["labels"], _ = filter_labels(dev_df, None, allowed=allowed_labels)
_, test_df["labels"], _ = filter_labels(test_df, None, allowed=allowed_labels)
dev_df = dev_df[dev_df["labels"].apply(len) > 0].reset_index(drop=True)
test_df = test_df[test_df["labels"].apply(len) > 0].reset_index(drop=True)

label_list = sorted(allowed_labels)
label2idx = {l: i for i, l in enumerate(label_list)}

with open("label_set.json", "w") as f:
    json.dump({"labels": label_list, "label2idx": label2idx}, f, indent=2)

print("=" * 50)
print("FINAL LABEL COUNT (post to the group NOW):", len(label_list))
print("=" * 50)

mlb = MultiLabelBinarizer(classes=label_list)
y_train = mlb.fit_transform(train_df["labels"])
y_dev   = mlb.transform(dev_df["labels"])
y_test  = mlb.transform(test_df["labels"])

**Freeze rule: don't touch `min_freq` again.** Every metric below depends on it staying fixed.

### 3. Reusable model functions (train once, call N times with different seeds)

In [ ]:
def run_rf(train_df, test_df, label_list, seed):
    mlb_local = MultiLabelBinarizer(classes=label_list)
    y_tr = mlb_local.fit_transform(train_df["labels"])
    y_te = mlb_local.transform(test_df["labels"])
    tfidf_local = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), sublinear_tf=True)
    X_tr = tfidf_local.fit_transform(train_df["text"])
    X_te = tfidf_local.transform(test_df["text"])
    rf_local = RandomForestClassifier(n_estimators=100, max_depth=20, n_jobs=RF_N_JOBS, random_state=seed)
    rf_local.fit(X_tr, y_tr)
    pred = rf_local.predict(X_te)
    return {
        "macro_f1": f1_score(y_te, pred, average="macro", zero_division=0),
        "micro_f1": f1_score(y_te, pred, average="micro", zero_division=0),
        "jaccard": jaccard_score(y_te, pred, average="samples", zero_division=0),
    }

In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, num_labels, embed_dim=128, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True,
                             bidirectional=True, num_layers=1)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim * 2, num_labels)

    def forward(self, x):
        emb = self.embedding(x)
        out, (h_n, _) = self.lstm(emb)
        combined = torch.cat([h_n[-2], h_n[-1]], dim=1)
        return self.fc(self.dropout(combined))

def build_vocab(texts, min_freq=2):
    counts = Counter()
    for t in texts:
        counts.update(tokenize(t))
    vocab = {"<pad>": 0, "<unk>": 1}
    for w, c in counts.items():
        if c >= min_freq:
            vocab[w] = len(vocab)
    return vocab

def make_lstm_loaders(vocab, train_df, dev_df, test_df, y_train, y_dev, y_test, max_len=256, batch_size=32):
    def encode(text):
        ids = [vocab.get(w, 1) for w in tokenize(text)][:max_len]
        ids += [0] * (max_len - len(ids))
        return ids

    class LSTMDataset(Dataset):
        def __init__(self, texts, labels_matrix):
            self.texts = texts
            self.labels = labels_matrix
        def __len__(self):
            return len(self.texts)
        def __getitem__(self, idx):
            ids = torch.tensor(encode(self.texts.iloc[idx]), dtype=torch.long)
            y = torch.tensor(self.labels[idx], dtype=torch.float)
            return ids, y

    train_loader = DataLoader(LSTMDataset(train_df["text"], y_train), batch_size=batch_size, shuffle=True)
    dev_loader   = DataLoader(LSTMDataset(dev_df["text"], y_dev), batch_size=64)
    test_loader  = DataLoader(LSTMDataset(test_df["text"], y_test), batch_size=64)
    return train_loader, dev_loader, test_loader

def get_probs(model, loader):
    model.eval()
    all_probs = []
    with torch.no_grad():
        for x_batch, _ in loader:
            x_batch = x_batch.to(DEVICE)
            probs = torch.sigmoid(model(x_batch)).cpu().numpy()
            all_probs.append(probs)
    return np.concatenate(all_probs)

def run_bilstm(vocab, loaders, label_list, y_dev, y_test, seed, epochs=6):
    torch.manual_seed(seed)
    train_loader, dev_loader, test_loader = loaders
    model = BiLSTMClassifier(len(vocab), len(label_list)).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        model.train()
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x_batch), y_batch)
            loss.backward()
            optimizer.step()

    dev_probs = get_probs(model, dev_loader)
    best_t, best_f1 = 0.5, -1
    for t in np.arange(0.2, 0.61, 0.05):
        preds = (dev_probs >= t).astype(int)
        f1 = f1_score(y_dev, preds, average="macro", zero_division=0)
        if f1 > best_f1:
            best_t, best_f1 = t, f1

    test_probs = get_probs(model, test_loader)
    test_preds = (test_probs >= best_t).astype(int)
    return {
        "macro_f1": f1_score(y_test, test_preds, average="macro", zero_division=0),
        "micro_f1": f1_score(y_test, test_preds, average="micro", zero_division=0),
        "precision": precision_score(y_test, test_preds, average="macro", zero_division=0),
        "recall": recall_score(y_test, test_preds, average="macro", zero_division=0),
        "jaccard": jaccard_score(y_test, test_preds, average="samples", zero_division=0),
    }, model, best_t

In [ ]:
def run_distilbert(train_df, dev_df, test_df, label_list, y_train, y_dev, y_test,
                    seed, epochs=2, max_len=256, save_dir=None):
    tokenizer_local = DistilBertTokenizerFast.from_pretrained(DISTILBERT_NAME)

    class FIRDataset(torch.utils.data.Dataset):
        def __init__(self, texts, labels_matrix):
            self.encodings = tokenizer_local(list(texts), truncation=True, padding="max_length", max_length=max_len)
            self.labels = labels_matrix.astype(np.float32)
        def __len__(self):
            return len(self.labels)
        def __getitem__(self, idx):
            item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
            item["labels"] = torch.tensor(self.labels[idx])
            return item

    train_ds = FIRDataset(train_df["text"], y_train)
    dev_ds   = FIRDataset(dev_df["text"], y_dev)
    test_ds  = FIRDataset(test_df["text"], y_test)

    model_local = DistilBertForSequenceClassification.from_pretrained(
        DISTILBERT_NAME,
        num_labels=len(label_list),
        problem_type="multi_label_classification",
    ).to(DEVICE)

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        probs = 1 / (1 + np.exp(-logits))
        preds = (probs >= 0.5).astype(int)
        return {"macro_f1": f1_score(labels, preds, average="macro", zero_division=0)}

    args = TrainingArguments(
        output_dir="./tmp_run",
        num_train_epochs=epochs,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=3e-5,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=200,
        report_to="none",
        seed=seed,
    )

    trainer_local = Trainer(
        model=model_local, args=args, train_dataset=train_ds,
        eval_dataset=dev_ds, compute_metrics=compute_metrics,
    )
    trainer_local.train()

    dev_logits = trainer_local.predict(dev_ds).predictions
    dev_probs = 1 / (1 + np.exp(-dev_logits))
    best_t, best_f1 = 0.5, -1
    for t in np.arange(0.2, 0.61, 0.05):
        preds = (dev_probs >= t).astype(int)
        f1 = f1_score(y_dev, preds, average="macro", zero_division=0)
        if f1 > best_f1:
            best_t, best_f1 = t, f1

    test_logits = trainer_local.predict(test_ds).predictions
    test_probs = 1 / (1 + np.exp(-test_logits))
    test_preds = (test_probs >= best_t).astype(int)
    metrics = {
        "macro_f1": f1_score(y_test, test_preds, average="macro", zero_division=0),
        "micro_f1": f1_score(y_test, test_preds, average="micro", zero_division=0),
        "jaccard": jaccard_score(y_test, test_preds, average="samples", zero_division=0),
        "precision": precision_score(y_test, test_preds, average="macro", zero_division=0),
        "recall": recall_score(y_test, test_preds, average="macro", zero_division=0),
    }

    if save_dir:
        trainer_local.save_model(save_dir)
        tokenizer_local.save_pretrained(save_dir)
        with open(os.path.join(save_dir, "threshold.json"), "w") as f:
            json.dump({"threshold": float(best_t)}, f)

    return metrics, trainer_local, tokenizer_local, best_t

### 4. Run each model N times on ILSI and average

**The point of this whole exercise is to gauge DistilBERT vs BiLSTM properly** — a single run
of each is just one noisy sample; running each N times and comparing the *distributions* of
macro-F1 (not just one number each) is what actually tells you which one is reliably better,
and by how much. That means BiLSTM and DistilBERT need to use the **same N** — an unequal
comparison (BiLSTM x10 vs DistilBERT x3) isn't a fair fight and won't hold up if the faculty
asks about it directly.

RF is only there as a reference baseline, not the thing being compared — it doesn't need N to
match the other two.

**Time budget, be honest about this:** DistilBERT at `epochs=2` takes ~8-15 min/run on a T4.
10 runs = ~1.5-2.5 hrs. Two ways to fit that in less time, pick one and say which you picked:
- Drop to `epochs=1` (cuts DistilBERT time roughly in half, some macro_f1 loss expected)
- Just run it overnight / while you sleep

`N_RUNS` below is shared by BOTH BiLSTM and DistilBERT so the comparison stays fair.

In [ ]:
# ===== RUN SETTINGS (lab) =====
N_RUNS = 10             # BiLSTM and DistilBERT -- keep equal for a fair comparison
DISTILBERT_EPOCHS = 2
BILSTM_EPOCHS = 6       # run_bilstm default
RF_N_RUNS = 10          # faculty asked for >= 10 runs per model
RF_N_JOBS = -1          # all CPU cores; if RF ever stalls with no output, set this to 1 and re-run
N_DISTILBERT_RUNS = N_RUNS   # used by the comparison-table cell further down

print(f"ILSI: {len(train_df)} train docs, {len(label_list)} labels")

# --- RF (reference baseline) ---
rf_runs = run_with_ckpt("ilsi_rf", RF_N_RUNS,
                        lambda s: run_rf(train_df, test_df, label_list, seed=s), "RF")
rf_avg = average_runs(rf_runs)
print_avg(f"TF-IDF+RF ({RF_N_RUNS} runs)", rf_avg)

In [ ]:
# --- BiLSTM (the model actually being compared against DistilBERT) ---
vocab = build_vocab(train_df["text"])
lstm_loaders = make_lstm_loaders(vocab, train_df, dev_df, test_df, y_train, y_dev, y_test)

lstm_runs = run_with_ckpt(
    f"ilsi_bilstm_ep{BILSTM_EPOCHS}", N_RUNS,
    lambda s: run_bilstm(vocab, lstm_loaders, label_list, y_dev, y_test, seed=s, epochs=BILSTM_EPOCHS)[0],
    "BiLSTM")
lstm_avg = average_runs(lstm_runs)
print_avg(f"BiLSTM ({N_RUNS} runs)", lstm_avg)

In [ ]:
# --- DistilBERT (the other half of the comparison) ---
SAVE_DIR = "./fir_distilbert_model"  # last run gets saved here for Track B
DB_TAG = f"ilsi_distilbert_ep{DISTILBERT_EPOCHS}"

# the last run saves the model for Track B; if its checkpoint exists but the saved model doesn't, redo that run
if not os.path.exists(os.path.join(SAVE_DIR, "threshold.json")):
    ckpt_forget(DB_TAG, N_RUNS - 1)

def _ilsi_distilbert(s):
    save_this_run = SAVE_DIR if s == N_RUNS - 1 else None  # only save the final run
    return run_distilbert(
        train_df, dev_df, test_df, label_list, y_train, y_dev, y_test,
        seed=s, epochs=DISTILBERT_EPOCHS, save_dir=save_this_run,
    )[0]

distilbert_runs = run_with_ckpt(DB_TAG, N_RUNS, _ilsi_distilbert, "DistilBERT")
distilbert_avg = average_runs(distilbert_runs)
print_avg(f"DistilBERT ({N_RUNS} runs)", distilbert_avg)

import shutil
shutil.copy("label_set.json", SAVE_DIR + "/label_set.json")
print("Final DistilBERT run saved to", SAVE_DIR, "- this is what goes to Track B")

### 5. `predict()` — from the saved (final-run) model, for Zaki

In [ ]:
MAX_LEN = 256

_pred_tokenizer = DistilBertTokenizerFast.from_pretrained(SAVE_DIR)
_pred_model = DistilBertForSequenceClassification.from_pretrained(SAVE_DIR).to(DEVICE)
_pred_model.eval()
with open(os.path.join(SAVE_DIR, "threshold.json")) as f:
    _threshold = json.load(f)["threshold"]

def predict(text, threshold=None):
    threshold = threshold if threshold is not None else _threshold
    inputs = _pred_tokenizer(text, truncation=True, padding="max_length", max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        logits = _pred_model(**inputs).logits.cpu().numpy()[0]
    probs = 1 / (1 + np.exp(-logits))
    results = [
        {"code": label_list[i], "confidence": float(probs[i])}
        for i in range(len(label_list)) if probs[i] >= threshold
    ]
    results.sort(key=lambda r: -r["confidence"])
    return results

print(predict(test_df["text"].iloc[0]))

### 6. Averaged comparison table for ILSI (mean ± std, for the deck)

In [ ]:
def avg_to_row(name, avg, n_runs):
    row = {"model": name, "n_runs": n_runs}
    for k, v in avg.items():
        row[k] = f"{v['mean']:.4f} ± {v['std']:.4f}"
    return row

ilsi_comparison = pd.DataFrame([
    avg_to_row("TF-IDF + RandomForest", rf_avg, RF_N_RUNS),
    avg_to_row("BiLSTM", lstm_avg, N_RUNS),
    avg_to_row("DistilBERT (fine-tuned)", distilbert_avg, N_DISTILBERT_RUNS),
])
ilsi_comparison.to_csv("ilsi_comparison_table.csv", index=False)
print("=== ILSI dataset: averaged comparison ===")
print(ilsi_comparison.to_string(index=False))

### 6b. BiLSTM vs DistilBERT — which one actually wins?

Two means with error bars isn't a verdict by itself — a Welch's t-test on the N per-run
macro-F1 scores tells you whether the gap between BiLSTM and DistilBERT is bigger than run-to-run
noise, or whether it could just be luck of the seed. This is the number to put in the deck next
to the bar chart.

In [ ]:
from scipy import stats

lstm_scores = [r["macro_f1"] for r in lstm_runs]
distilbert_scores = [r["macro_f1"] for r in distilbert_runs]

t_stat, p_val = stats.ttest_ind(distilbert_scores, lstm_scores, equal_var=False)

winner = "DistilBERT" if np.mean(distilbert_scores) > np.mean(lstm_scores) else "BiLSTM"
significant = p_val < 0.05

print(f"BiLSTM      macro_f1: {np.mean(lstm_scores):.4f} \u00b1 {np.std(lstm_scores):.4f}  (n={len(lstm_scores)})")
print(f"DistilBERT  macro_f1: {np.mean(distilbert_scores):.4f} \u00b1 {np.std(distilbert_scores):.4f}  (n={len(distilbert_scores)})")
print(f"Welch's t-test: t={t_stat:.3f}, p={p_val:.4f}")
print()
if significant:
    print(f"VERDICT: {winner} is significantly better on ILSI (p < 0.05).")
else:
    print(f"VERDICT: {winner} scored higher on average, but the gap is NOT statistically "
          f"significant at p<0.05 (p={p_val:.4f}) -- with only {len(lstm_scores)} runs each, "
          f"you can't rule out that this is seed noise. Say this honestly if asked: "
          f"'{winner} trended higher but the difference wasn't statistically significant.'")

---
## DATASET 2: ACI (Automatic Charge Identification, COLING 2020, same research group)

189 documents total (120 train / 69 test), charges given as **names** (e.g. "murder", "cheating")
rather than IPC section numbers, no login required. We carve 20% off the train split for a dev
set (needed for threshold tuning, same as ILSI). Given the tiny size, run all 3 models with the
same reusable functions above — just swap in this dataset's `train_df`/`dev_df`/`test_df`/`label_list`.

In [ ]:
ACI_BASE = "https://raw.githubusercontent.com/Law-AI/automatic-charge-identification/main/data/"

def load_jsonl_url(url):
    import urllib.request
    rows = []
    with urllib.request.urlopen(url) as resp:
        for line in resp:
            line = line.decode("utf-8").strip()
            if line:
                rows.append(json.loads(line))
    return rows

ACI_DIR = "./data/ACI"

def load_aci(fname):
    local = os.path.join(ACI_DIR, fname)
    return load_jsonl(local) if os.path.exists(local) else load_jsonl_url(ACI_BASE + fname)

aci_train_raw = load_aci("Train-Sent.jsonl")
aci_test_raw  = load_aci("Test-Doc.jsonl")

def aci_to_df(raw):
    return pd.DataFrame({
        "id": [r["factid"] for r in raw],
        "text": [" ".join(r["text"]) for r in raw],
        "labels": [r.get("doc_labels", []) for r in raw],
    })

aci_train_full = aci_to_df(aci_train_raw)
aci_test_df    = aci_to_df(aci_test_raw)

# carve a dev split (20%) off train, shuffled with a fixed seed so it's reproducible
aci_train_full = aci_train_full.sample(frac=1, random_state=42).reset_index(drop=True)
n_dev = max(1, int(0.2 * len(aci_train_full)))
aci_dev_df   = aci_train_full.iloc[:n_dev].reset_index(drop=True)
aci_train_df = aci_train_full.iloc[n_dev:].reset_index(drop=True)

print("ACI docs -> train:", len(aci_train_df), " dev:", len(aci_dev_df), " test:", len(aci_test_df))

aci_label_counts = Counter(l for labs in aci_train_full["labels"] for l in labs)
aci_label_list = sorted(aci_label_counts.keys())
print("ACI label count:", len(aci_label_list), aci_label_list)

aci_mlb = MultiLabelBinarizer(classes=aci_label_list)
aci_y_train = aci_mlb.fit_transform(aci_train_df["labels"])
aci_y_dev   = aci_mlb.transform(aci_dev_df["labels"])
aci_y_test  = aci_mlb.transform(aci_test_df["labels"])

In [ ]:
ACI_N_RUNS = 10  # ACI is tiny, so even DistilBERT is fast here -- 10 runs is realistic

# --- RF on ACI ---
aci_rf_runs = run_with_ckpt("aci_rf", ACI_N_RUNS,
                            lambda s: run_rf(aci_train_df, aci_test_df, aci_label_list, seed=s), "[ACI] RF")
aci_rf_avg = average_runs(aci_rf_runs)
print_avg(f"[ACI] TF-IDF+RF ({ACI_N_RUNS} runs)", aci_rf_avg)

In [ ]:
# --- BiLSTM on ACI ---
aci_vocab = build_vocab(aci_train_df["text"], min_freq=1)  # min_freq=1: ACI is small, can't afford to drop words
aci_lstm_loaders = make_lstm_loaders(aci_vocab, aci_train_df, aci_dev_df, aci_test_df,
                                      aci_y_train, aci_y_dev, aci_y_test, max_len=256, batch_size=8)

aci_lstm_runs = run_with_ckpt(
    "aci_bilstm_ep10", ACI_N_RUNS,
    lambda s: run_bilstm(aci_vocab, aci_lstm_loaders, aci_label_list, aci_y_dev, aci_y_test, seed=s, epochs=10)[0],
    "[ACI] BiLSTM")
aci_lstm_avg = average_runs(aci_lstm_runs)
print_avg(f"[ACI] BiLSTM ({ACI_N_RUNS} runs)", aci_lstm_avg)

In [ ]:
# --- DistilBERT on ACI ---
aci_distilbert_runs = run_with_ckpt(
    "aci_distilbert_ep4", ACI_N_RUNS,
    lambda s: run_distilbert(
        aci_train_df, aci_dev_df, aci_test_df, aci_label_list,
        aci_y_train, aci_y_dev, aci_y_test, seed=s, epochs=4, save_dir=None,
    )[0],
    "[ACI] DistilBERT")
aci_distilbert_avg = average_runs(aci_distilbert_runs)
print_avg(f"[ACI] DistilBERT ({ACI_N_RUNS} runs)", aci_distilbert_avg)

### 7. Combined summary — both datasets, for the deck

In [ ]:
aci_comparison = pd.DataFrame([
    avg_to_row("TF-IDF + RandomForest", aci_rf_avg, ACI_N_RUNS),
    avg_to_row("BiLSTM", aci_lstm_avg, ACI_N_RUNS),
    avg_to_row("DistilBERT (fine-tuned)", aci_distilbert_avg, ACI_N_RUNS),
])
aci_comparison.to_csv("aci_comparison_table.csv", index=False)

print("=== ILSI (100 IPC sections, N=see table) ===")
print(ilsi_comparison.to_string(index=False))
print()
print("=== ACI (20-24 charge names, N=10, smaller dataset) ===")
print(aci_comparison.to_string(index=False))

print("""
For the faculty / deck: DistilBERT should still come out on top on ILSI (the large dataset)
by macro-F1. ACI's numbers will likely be noisier (much smaller N=119 train docs), and that's
expected and worth saying out loud -- it demonstrates the pipeline transfers to a second,
independently-built Indian legal dataset without any architecture changes, which is the actual
generalisation claim, not a second production-scale benchmark.
""")

### 7b. Same verdict, on ACI

In [ ]:
aci_lstm_scores = [r["macro_f1"] for r in aci_lstm_runs]
aci_distilbert_scores = [r["macro_f1"] for r in aci_distilbert_runs]

t_stat_aci, p_val_aci = stats.ttest_ind(aci_distilbert_scores, aci_lstm_scores, equal_var=False)
winner_aci = "DistilBERT" if np.mean(aci_distilbert_scores) > np.mean(aci_lstm_scores) else "BiLSTM"
significant_aci = p_val_aci < 0.05

print(f"[ACI] BiLSTM      macro_f1: {np.mean(aci_lstm_scores):.4f} \u00b1 {np.std(aci_lstm_scores):.4f}")
print(f"[ACI] DistilBERT  macro_f1: {np.mean(aci_distilbert_scores):.4f} \u00b1 {np.std(aci_distilbert_scores):.4f}")
print(f"[ACI] Welch's t-test: t={t_stat_aci:.3f}, p={p_val_aci:.4f}")
if significant_aci:
    print(f"[ACI] VERDICT: {winner_aci} is significantly better (p < 0.05).")
else:
    print(f"[ACI] VERDICT: {winner_aci} trended higher, not statistically significant "
          f"(p={p_val_aci:.4f}) -- expected given ACI's small size (n=119 train docs).")

print()
print("Cross-dataset check: is the winner the SAME model on both ILSI and ACI?")
same_winner = winner == winner_aci
print(f"ILSI winner: {winner}  |  ACI winner: {winner_aci}  |  Consistent: {same_winner}")
if same_winner:
    print(f"{winner} wins on both datasets -- this is your strongest evidence for the deck.")
else:
    print("Winners differ across datasets -- worth flagging honestly rather than picking whichever "
          "result you like; likely explained by ACI's tiny size making DistilBERT's larger "
          "parameter count harder to fit well.")

---
## 8. Explainability (optional, skip if short on time)

Runs on the final saved ILSI DistilBERT model.

In [ ]:
from transformers_interpret import MultiLabelClassificationExplainer

_pred_tokenizer.model_max_length = MAX_LEN
explainer = MultiLabelClassificationExplainer(_pred_model, _pred_tokenizer)

def explain(text, top_k=8):
    truncated = " ".join(text.split()[:200])  # avoid the >512-token crash on long complaints
    word_attributions = explainer(truncated)
    preds = predict(text)
    if not preds:
        return []
    top_label = preds[0]["code"]
    attrs = word_attributions.get(top_label, [])
    attrs_sorted = sorted(attrs, key=lambda x: -abs(x[1]))[:top_k]
    return [{"token": tok, "weight": round(float(w), 3)} for tok, w in attrs_sorted]

print(explain(test_df["text"].iloc[0]))

## Handoff checklist

- [ ] Posted final ILSI label count to the group chat
- [ ] Zipped `fir_distilbert_model/` (includes `threshold.json`, `label_set.json`) and sent to Zaki
- [ ] Sent the `predict()` cell to Zaki
- [ ] Posted `ilsi_comparison_table.csv` and `aci_comparison_table.csv` to Sarah for the deck
- [ ] Wrote down N_DISTILBERT_RUNS actually used (3 vs 10) — say this number honestly in the review
- [ ] `label_set.json` committed / shared — does not change again